# Module 16 — Assembling One Transformer Block

Every piece is built: multi-head attention (Modules 10-11) mixes
information across tokens, positional encoding (12) gives the model a
sense of order, layer norm (13) keeps activation scale under control,
residual connections (14) keep gradients flowing through depth, and the
feed-forward block (15) lets each token "think" independently after
attention. This module wires attention + layer norm + residuals +
feed-forward into one real **transformer block** — the exact unit Module
17 stacks to build nanoGPT.

Pre-norm convention (Module 14): normalize *before* each sublayer, add the
residual *after*:
```
x = x + Attention(LayerNorm(x))
x = x + FeedForward(LayerNorm(x))
```

## 1. Copied-in pieces from Modules 11 and 15 (not re-explained here)

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(Q, K, V, causal=True):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


def split_heads(t, num_heads):
    seq_len, d_model = t.shape
    d_k = d_model // num_heads
    return t.view(seq_len, num_heads, d_k).transpose(0, 1)


def merge_heads(t):
    num_heads, seq_len, d_k = t.shape
    return t.transpose(0, 1).contiguous().view(seq_len, num_heads * d_k)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal=True):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        Qh, Kh, Vh = split_heads(Q, self.num_heads), split_heads(K, self.num_heads), split_heads(V, self.num_heads)
        out, _ = scaled_dot_product_attention(Qh, Kh, Vh, causal=causal)
        return self.Wo(merge_heads(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


print("Attention and feed-forward pieces ready.")

## 2. The transformer block itself

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.ln1(x), causal=True)
        x = x + self.ffn(self.ln2(x))
        return x


torch.manual_seed(42)
d_model, num_heads, seq_len = 16, 4, 6
block = TransformerBlock(d_model, num_heads)

x = torch.randn(seq_len, d_model)
out = block(x)
print("input shape: ", x.shape)
print("output shape:", out.shape)
assert out.shape == x.shape, "a transformer block must preserve shape - that's what lets Module 17 stack many of them"
print("Shape preserved, as required for stacking.")

## 3. The real end-to-end test: does the whole block stay causal?

Modules 10-11 checked that attention *weights* were correctly masked. Now
check the thing that actually matters: does changing a **later** token in
the input ever change an **earlier** token's output, once everything —
attention, layer norm, residual, feed-forward — is wired together? For a
correct causal block, the answer must be no.

In [ ]:
x_original = torch.randn(seq_len, d_model)
out_original = block(x_original)

x_modified = x_original.clone()
x_modified[3] = torch.randn(d_model) * 100  # drastically change a LATER token (position 3)
out_modified = block(x_modified)

for pos in range(seq_len):
    changed = not torch.allclose(out_original[pos], out_modified[pos], atol=1e-6)
    expected_to_change = pos >= 3
    assert changed == expected_to_change, f"position {pos}: changed={changed}, expected={expected_to_change}"
    print(f"position {pos}: output changed = {changed} (expected {expected_to_change})")

print("\nConfirmed: perturbing position 3 only affects positions 3 and later - the whole assembled block is correctly causal, not just the raw attention weights.")

## 4. Gradients flow end to end

In [ ]:
x_grad_test = torch.randn(seq_len, d_model, requires_grad=True)
out_grad_test = block(x_grad_test)
out_grad_test.sum().backward()
assert x_grad_test.grad is not None and x_grad_test.grad.norm().item() > 0
print("Gradient reaches the block's input with nonzero norm:", x_grad_test.grad.norm().item())

## 5. Stacking more than one (a preview of Module 17)

In [ ]:
blocks = nn.Sequential(*[TransformerBlock(d_model, num_heads) for _ in range(4)])
stacked_out = blocks(x)
assert stacked_out.shape == x.shape
print("4 stacked blocks, shape still preserved:", stacked_out.shape)

## Recap

- One `TransformerBlock` = pre-norm attention with a residual, then
  pre-norm feed-forward with a residual.
- Verified the strongest correctness property that matters: perturbing a
  later token never changes an earlier token's output through the *entire*
  assembled block, not just the raw attention weights.
- Shape is preserved (`(seq_len, d_model)` in, same out), which is exactly
  what lets these blocks be stacked arbitrarily deep.

Module 17 stacks several of these blocks together with a token embedding,
positional encoding, and a final output projection — the complete nanoGPT
architecture.